# 11 - Manager Analytics Data Audit

- Audit coverage and reliability before manager-facing analytics.
- Use canonical product IDs and batched review aggregates.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.rag.config import load_project_config
from src.rag.analytics import (
    AnalyticsRepository,
    AnalyticsDataAuditor,
    AnalyticsService,
)

In [2]:
config = load_project_config(
    PROJECT_ROOT
)["analytics"]

audit_config = config["audit"]
aggregation_config = config[
    "aggregation"
]

repository = (
    AnalyticsRepository
    .from_project_root(
        PROJECT_ROOT
    )
)

print(
    "Product source:",
    repository.product_source,
)

print(
    "Canonical products:",
    f"{len(repository.products):,}",
)

Product source: products_search.parquet
Canonical products: 948,352


In [3]:
auditor = AnalyticsDataAuditor(
    repository=repository,
    ready_coverage=(
        audit_config[
            "ready_coverage"
        ]
    ),
    limited_coverage=(
        audit_config[
            "limited_coverage"
        ]
    ),
    minimum_comment_join_rate=(
        audit_config[
            "minimum_comment_join_rate"
        ]
    ),
    product_rating_max=(
        audit_config[
            "product_rating_max"
        ]
    ),
    review_cap_min_products=(
        audit_config[
            "review_cap_min_products"
        ]
    ),
    generic_brand_values=(
        audit_config[
            "generic_brand_values"
        ]
    ),
    unknown_category_values=(
        audit_config[
            "unknown_category_values"
        ]
    ),
)

audit = auditor.run(
    top_n=20
)

REPORT_DIR = (
    PROJECT_ROOT
    / "data"
    / "evaluation"
    / "analytics"
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

audit.save_json(
    REPORT_DIR
    / "analytics_data_audit.json"
)

audit.summary

{'product_source': 'products_search.parquet',
 'canonical_product_count': 948352,
 'unique_product_ids': 948352,
 'duplicate_product_ids': 0,
 'total_comment_rows': 6153060,
 'matched_comment_rows': 6153060,
 'comment_product_join_rate': 1.0,
 'products_with_reviews': 331599,
 'product_review_coverage': 0.3496581438115805}

## Text fields

- Measure coverage, placeholders, and usable values.

In [4]:
display(
    audit.column_quality.sort_values(
        "usable_coverage",
        ascending=False,
    )
)

,field,rows,non_empty_count,coverage,unique_values,placeholder_count,placeholder_share,usable_coverage
0,title_fa,948352,948352,1.0,901043,0,0.000000,1.000000
2,Category1,948352,948352,1.0,228,0,0.000000,1.000000
4,sub_category,948352,948352,1.0,6,0,0.000000,1.000000
3,Category2,948352,948352,1.0,602,181346,0.191222,0.808778
1,Brand,948352,948352,1.0,8956,530051,0.558918,0.441082


## Numeric fields

- Measure usable coverage and numeric ranges.

In [5]:
display(
    audit.numeric_quality
)

,field,raw_valid_count,raw_coverage,usable_count,coverage,zero_count,negative_count,out_of_range_count,scale,min,p25,median,p75,p95,max
0,Price,948196,0.999836,948196,0.999836,0,0,0,NaN,20000.0,740000.0,1700000.0,4580000.0,34800000.0,8.499990e+09
1,min_price_last_month,55580,0.058607,55580,0.058607,0,0,0,NaN,10000.0,619000.0,1488000.0,4096000.0,28690000.0,7.645500e+09
2,Rate,948352,1.000000,360989,0.380649,587378,0,0,0-100,0.0,74.0,80.0,90.0,100.0,1.000000e+02
3,Rate_cnt,948352,1.000000,948352,1.000000,587363,0,0,NaN,0.0,0.0,0.0,2.0,40.0,3.043800e+04


## Review linkage

- Validate comment-to-product joins and review rating coverage.

In [6]:
display(
    audit.review_quality.T
)

,0
source,/home/ali/Desktop/projects/digikala-ai-assista...
total_comment_rows,6153060
valid_product_id_rows,6153060
matched_product_rows,6153060
orphan_product_rows,0
product_join_rate,1.0
valid_review_rate_rows,5618897
review_rate_coverage,0.913187
max_reviews_per_product_in_corpus,233
products_at_max_review_count,1


## KPI readiness

- Classify metrics as ready, limited, or unavailable.

In [7]:
display(
    audit.metric_readiness
)

print()
print(
    "Ready:",
    audit.metric_readiness[
        audit.metric_readiness[
            "status"
        ]
        == "ready"
    ]["metric"].tolist(),
)

print(
    "Limited:",
    audit.metric_readiness[
        audit.metric_readiness[
            "status"
        ]
        == "limited"
    ]["metric"].tolist(),
)

print(
    "Unavailable:",
    audit.metric_readiness[
        audit.metric_readiness[
            "status"
        ]
        == "unavailable"
    ]["metric"].tolist(),
)

,metric,status,coverage,reason
0,product_count,ready,1.000000,Canonical product IDs are the counting unit.
1,current_price_statistics,ready,0.999836,Price usable coverage.
2,product_rating_statistics,limited,0.380649,Rated-product coverage on the native 0..100 pr...
3,historical_price_statistics,unavailable,0.058607,min_price_last_month usable coverage.
4,category1_analysis,ready,1.000000,Category1 usable non-placeholder coverage.
5,category2_analysis,ready,0.808778,Category2 usable non-placeholder coverage.
6,sub_category_analysis,ready,1.000000,sub_category usable non-placeholder coverage.
7,brand_analysis,limited,0.441082,Usable brand coverage after excluding generic/...
8,review_presence_and_coverage,ready,1.000000,Share of valid comment product IDs that join t...
9,review_volume_ranking,ready,1.000000,No deterministic per-product review-count cap ...



Ready: ['product_count', 'current_price_statistics', 'category1_analysis', 'category2_analysis', 'sub_category_analysis', 'review_presence_and_coverage', 'review_volume_ranking', 'review_rating_statistics']
Limited: ['product_rating_statistics', 'brand_analysis']
Unavailable: ['historical_price_statistics']


## Category and brand distribution

- Inspect dominant catalog groups and brand coverage.

In [8]:
print("Top Category1")
display(
    audit.top_category1.head(15)
)

print("Top Category2")
display(
    audit.top_category2.head(20)
)

print("Top Brands")
display(
    audit.top_brands.head(20)
)

Top Category1


,Category1,product_count,product_share
0,لباس زنانه,77928,0.082172
1,اکسسوری زنانه,57514,0.060646
2,لباس مردانه,55970,0.059018
3,اکسسوری زنانه و مردانه,52644,0.055511
4,اسباب بازی,42645,0.044967
5,دخترانه,35260,0.037180
6,زیورآلات طلا زنانه,32903,0.034695
7,اکسسوری مردانه,32337,0.034098
8,کتاب چاپی غیر فارسی,31792,0.033523
9,دفتر و کاغذ و مقوا,30065,0.031702


Top Category2


,Category2,product_count,product_share
0,Unknown,181346,0.191222
1,دفتر,26458,0.027899
2,زیورآلات زنانه و مردانه,21925,0.023119
3,لباس دخترانه,21781,0.022967
4,فکری و آموزشی,20612,0.021735
5,زیورآلات زنانه,18045,0.019028
6,تی شرت مردانه,18044,0.019027
7,لباس پسرانه,17797,0.018766
8,زیورآلات نقره زنانه,15872,0.016736
9,گردنبند طلا زنانه,15743,0.016600


Top Brands


,Brand,product_count,product_share
0,متفرقه,530051,0.558918
1,لیردا,13798,0.014549
2,الن نار,12308,0.012978
3,خندالو,7687,0.008106
4,کرابو,7295,0.007692
5,اچ اند ام,7075,0.007460
6,ترمه ۱,6534,0.006890
7,اسمارا,5933,0.006256
8,27,5220,0.005504
9,کارانس,5202,0.005485


## Rating and review-volume semantics

- Treat product rating as a 0-100 score.
- Detect possible review-corpus truncation.

In [9]:
rate_quality = audit.numeric_quality[
    audit.numeric_quality[
        "field"
    ]
    == "Rate"
]

display(rate_quality)

review_diagnostics = (
    audit.review_quality[
        [
            column
            for column
            in [
                "max_reviews_per_product_in_corpus",
                "products_at_max_review_count",
                "products_at_max_with_rate_cnt_above_max",
                "review_count_cap_suspected",
                "review_rate_coverage",
            ]
            if column
            in audit.review_quality.columns
        ]
    ]
)

display(
    review_diagnostics.T
)

,field,raw_valid_count,raw_coverage,usable_count,coverage,zero_count,negative_count,out_of_range_count,scale,min,p25,median,p75,p95,max
2,Rate,948352,1.0,360989,0.380649,587378,0,0,0-100,0.0,74.0,80.0,90.0,100.0,100.0


,0
max_reviews_per_product_in_corpus,233
products_at_max_review_count,1
products_at_max_with_rate_cnt_above_max,1
review_count_cap_suspected,False
review_rate_coverage,0.913187


## Aggregation checks

- Run deterministic category and leaderboard calculations.

In [10]:
analytics = AnalyticsService(
    repository=repository,
    generic_brand_values=(
        audit_config[
            "generic_brand_values"
        ]
    ),
    unknown_category_values=(
        audit_config[
            "unknown_category_values"
        ]
    ),
    min_rating_count_for_leaders=(
        aggregation_config[
            "min_rating_count_for_leaders"
        ]
    ),
    default_top_n=(
        aggregation_config[
            "default_top_n"
        ]
    ),
    product_rating_max=(
        aggregation_config[
            "product_rating_max"
        ]
    ),
)

overall = analytics.overview()

pd.DataFrame(
    [
        {
            "product_count": (
                overall[
                    "product_count"
                ]
            ),
            "brand_count": (
                overall[
                    "brand_count"
                ]
            ),
            "review_count": (
                overall[
                    "review_count"
                ]
            ),
            "review_coverage": (
                overall[
                    "review_coverage"
                ]
            ),
            "median_price": (
                overall[
                    "price"
                ][
                    "median"
                ]
            ),
            "price_coverage": (
                overall[
                    "price"
                ][
                    "coverage"
                ]
            ),
            "weighted_product_rating": (
                overall[
                    "weighted_product_rating"
                ]
            ),
            "weighted_review_rating": (
                overall[
                    "weighted_review_rating"
                ]
            ),
        }
    ]
)

,product_count,brand_count,review_count,review_coverage,median_price,price_coverage,weighted_product_rating,weighted_review_rating
0,948352,8955,6153060,0.349658,1700000.0,0.999836,81.591119,3.994147


In [11]:
category2_table = (
    analytics.category_table(
        category_field="Category2",
        top_n=20,
        include_unknown=False,
    )
)

display(
    category2_table
)

,Category2,brand_count,product_count,review_count,products_with_reviews,review_coverage,median_price,avg_rating,rated_product_count,rated_product_coverage,weighted_product_rating,rating_count_total
0,دفتر,85,26458,82320,5928,0.224053,660000.0,85.978703,7137,0.269748,86.238371,213021
1,زیورآلات زنانه و مردانه,78,21925,59679,5208,0.237537,600000.0,79.721421,5406,0.246568,80.424989,129401
2,لباس دخترانه,279,21781,40525,5169,0.237317,1700000.0,78.508392,5779,0.265323,78.321594,65051
3,فکری و آموزشی,290,20612,215226,9499,0.460848,1250000.0,79.101839,9299,0.451145,80.734273,441073
4,زیورآلات زنانه,119,18045,51676,3697,0.204877,1000000.0,79.074535,5219,0.289221,79.635330,149143
5,تی شرت مردانه,187,18044,58911,5239,0.290346,2299000.0,70.421135,5110,0.283197,66.263000,162380
6,لباس پسرانه,243,17797,44046,4715,0.264932,1680000.0,78.066593,5406,0.303759,78.195269,65059
7,زیورآلات نقره زنانه,46,15872,12068,3042,0.191658,3257500.0,74.068687,3465,0.218309,74.170941,15245
8,گردنبند طلا زنانه,24,15743,0,0,0.000000,41200000.0,71.565079,630,0.040018,72.777828,4438
9,لباس زیر زنانه,188,12930,80764,5322,0.411601,1200000.0,77.520701,5362,0.414695,77.383518,222631


In [12]:
smoke_categories = (
    category2_table[
        "Category2"
    ]
    .head(3)
    .tolist()
)

print(
    "Smoke categories:",
    smoke_categories,
)

display(
    analytics.compare_categories(
        category_values=(
            smoke_categories
        ),
        category_field="Category2",
    )
)

Smoke categories: ['دفتر', 'زیورآلات زنانه و مردانه', 'لباس دخترانه']


,Category2,product_count,brand_count,review_count,review_coverage,median_price,price_coverage,avg_product_rating,weighted_product_rating,rating_count_total,weighted_review_rating
0,دفتر,26458,85,82320,0.224053,660000.0,0.999698,85.978703,86.238371,213021,4.310039
1,زیورآلات زنانه و مردانه,21925,78,59679,0.237537,600000.0,1.000000,79.721421,80.424989,129401,4.000983
2,لباس دخترانه,21781,279,40525,0.237317,1700000.0,1.000000,78.508392,78.321594,65051,3.833497


In [13]:
if smoke_categories:
    smoke_category = (
        smoke_categories[0]
    )

    smoke_filters = {
        "Category2": (
            smoke_category
        )
    }

    print(
        "Category:",
        smoke_category,
    )

    display(
        analytics.top_brands(
            filters=(
                smoke_filters
            ),
            top_n=10,
            include_generic=False,
        )
    )

    print(
        "Most reviewed products"
    )

    display(
        analytics.top_products(
            filters=(
                smoke_filters
            ),
            sort_by=(
                "review_count"
            ),
            top_n=10,
        )
    )

    print(
        "Top rated products "
        "(minimum rating-count guard)"
    )

    display(
        analytics.top_products(
            filters=(
                smoke_filters
            ),
            sort_by="rating",
            top_n=10,
        )
    )

Category: دفتر


,Brand,product_count,review_count,products_with_reviews,review_coverage,median_price,avg_rating,rated_product_count,rated_product_coverage,weighted_product_rating,rating_count_total,product_share
0,مشایخ,758,4126,454,0.598945,550000.0,86.869748,476,0.627968,86.884987,5695,0.028649
1,پاپکو,434,4903,227,0.523041,615500.0,87.416667,288,0.663594,86.990324,14365,0.016403
2,کارنیلا,419,33,21,0.050119,850000.0,94.375000,16,0.038186,93.684211,19,0.015836
3,الیپون,312,3407,245,0.785256,444500.0,88.041667,240,0.769231,89.660369,4287,0.011792
4,سم,308,651,111,0.360390,620000.0,90.484848,99,0.321429,89.180396,959,0.011641
5,بنی دکو,303,843,110,0.363036,752000.0,83.053571,112,0.369637,86.994004,1501,0.011452
6,مستر راد,242,17570,218,0.900826,700000.0,87.582222,225,0.929752,87.243214,45791,0.009147
7,هیلا,184,412,103,0.559783,354950.0,85.098901,91,0.494565,85.345361,388,0.006954
8,حس آمیزی,178,0,0,0.000000,970000.0,NaN,0,0.000000,NaN,0,0.006728
9,آلما,176,35,27,0.153409,620000.0,81.263158,19,0.107955,88.352941,51,0.006652


Most reviewed products


,id,title_fa,Brand,Category1,Category2,sub_category,Price,Rate,Rate_cnt,review_count,avg_review_rate
0,1969602,دفتر مشق 50 برگ مدل BS 209,متفرقه,دفتر و کاغذ و مقوا,دفتر,book & stationary & art,270000.0,84,11413,200,4.246231
1,1918899,دفتر مشق 100 برگ کد M1,متفرقه,دفتر و کاغذ و مقوا,دفتر,book & stationary & art,414700.0,86,10806,200,4.24
2,3356888,دفتر برنامه ریزی طرح یونیکورن کد 001,متفرقه,دفتر و کاغذ و مقوا,دفتر,book & stationary & art,850000.0,90,3749,200,4.592965
3,2037856,دفترچه یادداشت آونگ کد 001,متفرقه,دفتر و کاغذ و مقوا,دفتر,book & stationary & art,180000.0,80,2930,200,3.82
4,5984252,دفترچه یادداشت مستر راد مدل to do list کد iPhone,مستر راد,دفتر و کاغذ و مقوا,دفتر,book & stationary & art,400000.0,86,2788,200,4.515
5,3230441,دفتر زبان 50 برگ کیهان کد K-1,متفرقه,دفتر و کاغذ و مقوا,دفتر,book & stationary & art,370000.0,88,2756,200,4.49
6,3739821,دفترچه یادداشت 100 برگ پاپکو مدل متالیک 1,پاپکو,دفتر و کاغذ و مقوا,دفتر,book & stationary & art,522000.0,90,2342,200,4.61
7,1961032,دفتر مشق 80 برگ مدل 201,متفرقه,دفتر و کاغذ و مقوا,دفتر,book & stationary & art,393000.0,86,2329,200,4.29
8,2280472,دفتر مشق 100 برگ بوفی مدل A-100,متفرقه,دفتر و کاغذ و مقوا,دفتر,book & stationary & art,330000.0,80,2301,200,4.0
9,8623307,دفتر برنامه ریزی مستر راد طرح اسب تک شاخ مدل پ...,مستر راد,دفتر و کاغذ و مقوا,دفتر,book & stationary & art,350000.0,82,2277,200,4.08


Top rated products (minimum rating-count guard)


,id,title_fa,Brand,Category1,Category2,sub_category,Price,Rate,Rate_cnt,review_count,avg_review_rate
0,462652,دفتر مشق 80 برگ پاپکو کد A4-603,پاپکو,دفتر و کاغذ و مقوا,دفتر,book & stationary & art,940000.0,100,10,0,NaN
1,7021502,دفتر یادداشت باژیکان مدل انیمه ناروتو کد 20111...,متفرقه,دفتر و کاغذ و مقوا,دفتر,book & stationary & art,888000.0,100,10,0,NaN
2,7748043,دفتر مشق 80 برگ گلبرگ طرح پسرانه بیچی مکس کد 948,متفرقه,دفتر و کاغذ و مقوا,دفتر,book & stationary & art,1000000.0,100,10,0,NaN
3,3555050,دفتر یادداشت مشایخ طرح استاد شجریان کد 5146,مشایخ,دفتر و کاغذ و مقوا,دفتر,book & stationary & art,450000.0,98,16,13,4.923077
4,6762810,دفتر یادداشت باژیکان مدل بلک پینک کد 2011158 ب...,متفرقه,دفتر و کاغذ و مقوا,دفتر,book & stationary & art,799000.0,98,14,10,4.9
5,11636334,دفتر نقاشی مستر راد مدل یونیکورن طرح الی کد fi...,مستر راد,دفتر و کاغذ و مقوا,دفتر,book & stationary & art,650000.0,98,13,58,4.534483
6,3107494,دفتر 80 برگ اسپادانا طرح یوز ایرانی کد PC3,متفرقه,دفتر و کاغذ و مقوا,دفتر,book & stationary & art,390000.0,98,12,9,4.888889
7,4278584,دفترچه یادداشت مدل یونیکورن تک شاخ کد 10 مجموع...,متفرقه,دفتر و کاغذ و مقوا,دفتر,book & stationary & art,270000.0,98,11,11,3.888889
8,3842448,دفتر مشق 50 برگ یاس بهشت مدل classic 03,متفرقه,دفتر و کاغذ و مقوا,دفتر,book & stationary & art,396000.0,98,10,6,4.833333
9,5396946,دفتر برنامه ریزی تیج سان مدل Magic Planner طرح...,تیج سان,دفتر و کاغذ و مقوا,دفتر,book & stationary & art,990000.0,98,10,0,NaN
